In [1]:
from gnncloudmanufacturing.data import read_fatahi_dataset
from gnncloudmanufacturing.random_solver import random_solve
from gnncloudmanufacturing.validation import total_cost_from_graph, check_feasibility, total_cost_from_gamma
from gnncloudmanufacturing.utils import delta_from_gamma, graph_from_problem, gamma_from_target, delta_from_gamma
from gnncloudmanufacturing.graph_model import GNN, os_type, ss_type

import numpy as np
from tqdm.auto import trange, tqdm
from time import time
import pandas as pd
import torch

In [2]:
def predict(model, dataset, n_operations):
    problem_name = []
    total_cost = []
    comp_time = []
    for problem in tqdm(dataset):
        start = time()
        total = np.inf
        for i in range(10):
            graph = graph_from_problem(problem, max_operations=n_operations)
            graph.edata['feat'][os_type][:, 0] /= 10
            graph.edata['feat'][ss_type][:] /= 100
            pred = model.predict(graph)
            gamma = gamma_from_target(pred, graph, problem)
            delta = delta_from_gamma(problem, gamma)
            check_feasibility(gamma, delta, problem)
            _total = total_cost_from_gamma(problem, gamma, delta).item()
            if total > _total:
                total = _total
        total_cost.append(total)
        problem_name.append(problem['name'])
        comp_time.append(time() - start)
    return pd.DataFrame({'problem_name': problem_name, 'total_cost': total_cost, 'comp_time': comp_time}).round(2)

In [3]:
n_tasks, n_operations, n_cities = 5, 5, 5
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,5,5-1
Problem: 5,5,5-2
Problem: 5,5,5-3


In [4]:
model = GNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=16,
    n_layers=1,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=16, bias=True)
      (W_os): Linear(in_features=7, out_features=16, bias=True)
      (W_ss): Linear(in_features=2, out_features=16, bias=True)
      (attn): Linear(in_features=32, out_features=1, bias=True)
      (W_in): Linear(in_features=5, out_features=16, bias=True)
      (W_self): Linear(in_features=5, out_features=16, bias=True)
      (W_out): Linear(in_features=5, out_features=16, bias=True)
      (W_o): Linear(in_features=48, out_features=16, bias=True)
    )
  )
  (dropout): Dropout(p=0.0, inplace=False)
  (dec): DotProductDecoder()
)

In [5]:
results_5_5_5 = predict(model, dataset, n_operations)
results_5_5_5

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"5,5,5-1",2914.58,0.07
1,"5,5,5-2",5944.42,0.06
2,"5,5,5-3",6653.88,0.06


In [6]:
n_tasks, n_operations, n_cities = 5, 10, 10
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,10,10-1
Problem: 5,10,10-2
Problem: 5,10,10-3


In [7]:
model = GNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=32,
    n_layers=3,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=32, bias=True)
      (W_os): Linear(in_features=12, out_features=32, bias=True)
      (W_ss): Linear(in_features=2, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=10, out_features=32, bias=True)
      (W_self): Linear(in_features=10, out_features=32, bias=True)
      (W_out): Linear(in_features=10, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
    (1-2): 2 x AttnConvLayer(
      (W_s): Linear(in_features=32, out_features=32, bias=True)
      (W_os): Linear(in_features=34, out_features=32, bias=True)
      (W_ss): Linear(in_features=33, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=32, out_features=32, bias=True)
      (W_self): Linear(in_features=32, out_features=32, bias=True)
    

In [8]:
results_5_10_10 = predict(model, dataset, n_operations)
results_5_10_10

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"5,10,10-1",6229.01,0.19
1,"5,10,10-2",7352.66,0.20
2,"5,10,10-3",7652.32,0.20


In [9]:
n_tasks, n_operations, n_cities = 10, 10, 10
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 10,10,10-1
Problem: 10,10,10-2
Problem: 10,10,10-3


In [10]:
model = GNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=32,
    n_layers=3,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=32, bias=True)
      (W_os): Linear(in_features=12, out_features=32, bias=True)
      (W_ss): Linear(in_features=2, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=10, out_features=32, bias=True)
      (W_self): Linear(in_features=10, out_features=32, bias=True)
      (W_out): Linear(in_features=10, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
    (1-2): 2 x AttnConvLayer(
      (W_s): Linear(in_features=32, out_features=32, bias=True)
      (W_os): Linear(in_features=34, out_features=32, bias=True)
      (W_ss): Linear(in_features=33, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=32, out_features=32, bias=True)
      (W_self): Linear(in_features=32, out_features=32, bias=True)
    

In [11]:
results_10_10_10 = predict(model, dataset, n_operations)
results_10_10_10

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"10,10,10-1",16438.31,0.28
1,"10,10,10-2",14275.53,0.29
2,"10,10,10-3",14200.75,0.30


In [12]:
n_tasks, n_operations, n_cities = 5, 10, 20
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,10,20-1
Problem: 5,10,20-2
Problem: 5,10,20-3


In [13]:
model = GNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=32,
    n_layers=3,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=32, bias=True)
      (W_os): Linear(in_features=12, out_features=32, bias=True)
      (W_ss): Linear(in_features=2, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=10, out_features=32, bias=True)
      (W_self): Linear(in_features=10, out_features=32, bias=True)
      (W_out): Linear(in_features=10, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
    (1-2): 2 x AttnConvLayer(
      (W_s): Linear(in_features=32, out_features=32, bias=True)
      (W_os): Linear(in_features=34, out_features=32, bias=True)
      (W_ss): Linear(in_features=33, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=32, out_features=32, bias=True)
      (W_self): Linear(in_features=32, out_features=32, bias=True)
    

In [14]:
results_5_10_20 = predict(model, dataset, n_operations)
results_5_10_20

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"5,10,20-1",4787.41,0.42
1,"5,10,20-2",5497.29,0.45
2,"5,10,20-3",5866.18,0.47


In [15]:
n_tasks, n_operations, n_cities = 5, 20, 10
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,20,10-1
Problem: 5,20,10-2
Problem: 5,20,10-3


In [16]:
model = GNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=32,
    n_layers=4,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=32, bias=True)
      (W_os): Linear(in_features=22, out_features=32, bias=True)
      (W_ss): Linear(in_features=2, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=20, out_features=32, bias=True)
      (W_self): Linear(in_features=20, out_features=32, bias=True)
      (W_out): Linear(in_features=20, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
    (1-3): 3 x AttnConvLayer(
      (W_s): Linear(in_features=32, out_features=32, bias=True)
      (W_os): Linear(in_features=34, out_features=32, bias=True)
      (W_ss): Linear(in_features=33, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=32, out_features=32, bias=True)
      (W_self): Linear(in_features=32, out_features=32, bias=True)
    

In [17]:
results_5_20_10 = predict(model, dataset, n_operations)
results_5_20_10

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"5,20,10-1",15707.66,0.33
1,"5,20,10-2",15915.55,0.33
2,"5,20,10-3",17143.11,0.34


In [18]:
n_tasks, n_operations, n_cities = 5, 20, 20
dataset = read_fatahi_dataset(
    '../../data/fatahi.xlsx', 
    sheet_names=[
        f'{n_tasks},{n_operations},{n_cities}-1',
        f'{n_tasks},{n_operations},{n_cities}-2', 
        f'{n_tasks},{n_operations},{n_cities}-3',
    ]
)
for problem in dataset:
    print(f'Problem: {problem["name"]}')

  0%|          | 0/3 [00:00<?, ?it/s]

Problem: 5,20,20-1
Problem: 5,20,20-2
Problem: 5,20,20-3


In [19]:
model = GNN.load_from_checkpoint(
    checkpoint_path=f'gnn-{n_tasks}-{n_operations}-{n_cities}.ckpt',
    ins_dim=1,
    ino_dim=n_operations,
    out_dim=32,
    n_layers=3,
    lr=0.002,
)
model.eval()

GNN(
  (convs): ModuleList(
    (0): AttnConvLayer(
      (W_s): Linear(in_features=1, out_features=32, bias=True)
      (W_os): Linear(in_features=22, out_features=32, bias=True)
      (W_ss): Linear(in_features=2, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=20, out_features=32, bias=True)
      (W_self): Linear(in_features=20, out_features=32, bias=True)
      (W_out): Linear(in_features=20, out_features=32, bias=True)
      (W_o): Linear(in_features=96, out_features=32, bias=True)
    )
    (1-2): 2 x AttnConvLayer(
      (W_s): Linear(in_features=32, out_features=32, bias=True)
      (W_os): Linear(in_features=34, out_features=32, bias=True)
      (W_ss): Linear(in_features=33, out_features=32, bias=True)
      (attn): Linear(in_features=64, out_features=1, bias=True)
      (W_in): Linear(in_features=32, out_features=32, bias=True)
      (W_self): Linear(in_features=32, out_features=32, bias=True)
    

In [20]:
results_5_20_20 = predict(model, dataset, n_operations)
results_5_20_20

  0%|          | 0/3 [00:00<?, ?it/s]

,problem_name,total_cost,comp_time
0,"5,20,20-1",13774.30,0.85
1,"5,20,20-2",14012.69,0.90
2,"5,20,20-3",15828.03,0.92


In [21]:
pd.concat([
    results_5_10_10,
    results_10_10_10,
    results_5_10_20,
    results_5_20_10,
    results_5_20_20,
    results_5_5_5,
]).reset_index(drop=True)

,problem_name,total_cost,comp_time
0,"5,10,10-1",6229.01,0.19
1,"5,10,10-2",7352.66,0.20
2,"5,10,10-3",7652.32,0.20
3,"10,10,10-1",16438.31,0.28
4,"10,10,10-2",14275.53,0.29
5,"10,10,10-3",14200.75,0.30
6,"5,10,20-1",4787.41,0.42
7,"5,10,20-2",5497.29,0.45
8,"5,10,20-3",5866.18,0.47
9,"5,20,10-1",15707.66,0.33
